In [ ]:
# install additional packages
!pip install gower
!pip install pyreadr

In [ ]:
# load packages
import gower
import pyreadr
from google.colab import files, data_table
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import ListedColormap
import seaborn as sns
import pickle
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.linear_model import BayesianRidge
from scipy import stats

# **Prepare dataset**

## **01 | Load the original data**

In [ ]:
# load data
DATA_PATH = "data/ACS_SexDiffereces_prepared_13092025.rds"
dfWholeDatasetORIGINAL = pyreadr.read_r(DATA_PATH)[None]

In [ ]:
# make copy to work on and keep original dataset
dfWholeDataset = dfWholeDatasetORIGINAL.copy()

## **02 | Prepare datasets, subset to variables of interest and remove rows with too many missing values**

**Remove patients with too many missing values**

In [ ]:
# subset to identical dataset
dfSelection = dfWholeDataset.copy()

# only allow rows with less than 20% NA values (meaning max. 2 columns with NA values)
print("Number of rows before removing rows with too many NAs:", dfSelection.shape[0])
threshold = len(dfSelection.columns) * 0.2
dfSelection = dfSelection[dfSelection.isna().sum(axis=1) <= threshold]
print("Number of rows after removing rows with too many NAs:", dfSelection.shape[0])

In [ ]:
# for each variable report percentage of missing values and round by three digits
missing_values = (dfSelection.isna().mean() * 100).round(3)
missing_values.sort_values(ascending=False)

**Conditionally add more variables**

In [ ]:
# add new column called "typical_atypical_noCP", with 2 if ThoraxschmerzNA=1, 1 if ThoraxschmerzatypischNA=1 and else 0
dfSelection['typical_atypical_noCP'] = dfSelection.apply(lambda row: 2 if row['ThoraxschmerzNA'] == 1
                                                                          else 1 if row['ThoraxschmerzatypischNA'] == 1
                                                                          else 0, axis=1)

# add new column called "sumOfRF"summing up risk factors (columns "Dyslipidämiebekannt", "FAAnamnesebekannt", "Nikotindichbekannt",	"Hypertonie",	"Diabetes")
dfSelection["sumOfRF"] = dfSelection[["Dyslipidämiebekannt", "FAAnamnesebekannt", "Nikotindichbekannt", "Hypertonie", "Diabetes"]].sum(axis=1)

# add new column called "ECG_ERBST", set to 1 if 1 in columns "EKGSTHebung"	"EKGSTSenkungen"	"EKGTneg"	"EKGLSB_any", else 0
dfSelection["ECG_ERBST"] = dfSelection[["EKGSTHebung", "EKGSTSenkungen", "EKGTneg", "EKGLSB_any"]].sum(axis=1)

# add new column called "Troponin_binary" with 1 if hsTrop+ or Trop+ in column "Biomarker"
dfSelection["Troponin_binary"] = dfSelection["Biomarker"].apply(lambda x: 1 if x in ["hsTrop+", "Trop+"] else 0)
dfSelection = dfSelection.drop(columns=["Biomarker"])

# convert column "TropAssay" to numerical, leave NA if empty
dfSelection["TropAssay"] = dfSelection["TropAssay"].map({"hsTrop": 1, "Trop": 0})

# calculate delta Trop
dfSelection["deltaTrop"] = dfSelection["Troponinwert2"] - dfSelection["Troponinwert"]

# calculate delta hsTrop
dfSelection["deltaHsTrop"] = dfSelection["hsTroponinwert2"] - dfSelection["hsTroponinwert"]

## **03 | Impute missing values**

In [ ]:
# define numeric, ordinal and categorical
                                                                                                                                               ### CAVE: Do not predict (hs)Troponin values!!!
num_cols = ["Alter", "Symptombeginn", "RRsysNA", "HFNA", "BMI", "CK", "LDH", "Lactat", "Hb", "Leukozyten", "CRP", "Krea"]                      # numeric
ord_cols = ["sumOfRF", "SchmerzNA", "typical_atypical_noCP"]                                                                                   # ordinal
cat_cols = ["Geschlecht", "EKGSTHebung", "EKGSTSenkungen", "EKGTneg", "EKGLSB_any", "ThoraxschmerzNA", "ThoraxschmerzatypischNA", "DyspnoeNA", "Dyslipidämiebekannt",
            "FAAnamnesebekannt", "Nikotindichbekannt", "Hypertonie", "Diabetes", "BekannteKHK", "GerinnungsmedNA", "BetaBlockerNA", "NitroNA", "AntieemeseNA", "MorphinNA",
            "Thoraxschmerz", "Thoraxschmerzatypisch", "Dyspnoe", "ChronischeNiereninsuffizienz", "EKG_STE_Klinik", "EKG_STD_Klinik", "EKG_TNeg_Klinik", "EKG_LSB_any_Klinik",
            "EchoLVEF", "EchoRWBST", "TropAssay", "InnerklinischeReanimation", "INtensivtherapie", "Todinnerklinisch", "OutcomeSTEMI", "OutcomeNSTEMI", "OutcomeKoronarverschluss",
            "OutcomeInterventionsbedarf", "MACE", "ECG_ERBST", "Troponin_binary"]                                                              # categoric

# Define the **order** for each ordinal column (from lowest to highest)
ordinal_orders = {
    "sumOfRF":   [0, 1, 2, 3, 4, 5],
    "SchmerzNA":   [0, 1, 2, 3, 4, 5, 6,7,8,9,10],
    "typical_atypical_noCP":   [0, 1, 2]
}

In [ ]:
df_work = dfSelection.copy()

# Encode ordinal columns (numbers only so MICE can handle them)
oe = OrdinalEncoder(categories=[ordinal_orders[c] for c in ord_cols],
                    handle_unknown="use_encoded_value", unknown_value=np.nan)

if len(ord_cols) > 0:
    # Fit on available (non-missing) labels; missing stays NaN
    df_work[ord_cols] = oe.fit_transform(df_work[ord_cols])

# MICE over numeric + encoded ordinal together
mice_cols = num_cols + ord_cols
mice = IterativeImputer(
    estimator=BayesianRidge(),        # solid default; can swap to ExtraTreesRegressor
    max_iter=10,
    sample_posterior=True,            # proper MICE-style stochasticity
    min_value=0.0,
    random_state=0
)
df_work[mice_cols] = mice.fit_transform(df_work[mice_cols])

# Round & clip ordinal back to valid levels, then decode to labels
for j, c in enumerate(ord_cols):
    # round to nearest valid code
    df_work[c] = np.rint(df_work[c])
    # clip to valid range [0, n_levels-1]
    n_levels = len(ordinal_orders[c])
    df_work[c] = df_work[c].clip(0, n_levels - 1)

if len(ord_cols) > 0:
    # inverse_transform needs a 2D array; then put back as Series
    decoded = oe.inverse_transform(df_work[ord_cols].to_numpy())
    df_work[ord_cols] = decoded

# Impute nominal (unordered) with most-frequent
if len(cat_cols) > 0:
    imp_nom = SimpleImputer(strategy="most_frequent")
    df_work[cat_cols] = imp_nom.fit_transform(df_work[cat_cols])

df_imputed = df_work  # final result

In [ ]:
# round to 0 digit:
df_imputed[["Alter", "Symptombeginn", "RRsysNA", "HFNA", "Leukozyten", "CK", "LDH"]] = df_imputed[["Alter", "Symptombeginn", "RRsysNA", "HFNA", "Leukozyten", "CK", "LDH"]].round(0)

# round to 1 digit:
df_imputed[["Lactat", "Hb", "Krea"]] = df_imputed[["Lactat", "Hb", "Krea"]].round(1)

# round to 2 digits:
df_imputed[["BMI", "CRP"]] = df_imputed[["BMI", "CRP"]].round(2)

# **Fig. 1: Gower Distance to identify ACS twins**

In [ ]:
# list all columns with index
for idx, col in enumerate(df_imputed.columns):
    print(f"Index {idx}: {col}")

In [ ]:
# select columns for Gower Distance patient twin analysis
df_gower = df_imputed.copy()
df_gower = df_gower.iloc[:, [0,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19]] # prehospital twins

#### **Identify digital twins with opposite sex**

In [ ]:
# Align indices
sex_raw = df_imputed["Geschlecht"].reindex(df_gower.index)

# Map 0/1 to {'female','male'} and drop anything not 0/1
sex = pd.to_numeric(sex_raw, errors='coerce').map({0: 'female', 1: 'male'})
mask_valid = sex.notna()

dfm = df_gower.loc[mask_valid].copy()
sex = sex.loc[mask_valid]
sex.value_counts()

**Split groups & choose which side to fully match**

In [ ]:
fem_idx = dfm.index[sex.eq('female')]
mal_idx = dfm.index[sex.eq('male')]

if len(fem_idx) == 0 or len(mal_idx) == 0:
    raise ValueError("Need at least one female and one male to match.")

# Match everyone in the smaller group (typical)
if len(fem_idx) <= len(mal_idx):
    left_idx, right_idx = fem_idx, mal_idx
    left_sex, right_sex = 'female', 'male'
else:
    left_idx, right_idx = mal_idx, fem_idx
    left_sex, right_sex = 'male', 'female'

**Build Gower distance & compute cross-sex distances**

In [ ]:
# 1) Validate cat_cols against dfm
cat_cols = ["Geschlecht", "EKGSTHebung", "EKGSTSenkungen", "EKGTneg", "EKGLSB_any", "ThoraxschmerzNA", "ThoraxschmerzatypischNA", "DyspnoeNA", "Dyslipidämiebekannt",
            "FAAnamnesebekannt", "Nikotindichbekannt", "Hypertonie", "Diabetes", "BekannteKHK", "GerinnungsmedNA", "BetaBlockerNA", "NitroNA", "AntieemeseNA", "MorphinNA",
            "Thoraxschmerz", "Thoraxschmerzatypisch", "Dyspnoe", "ChronischeNiereninsuffizienz", "EKG_STE_Klinik", "EKG_STD_Klinik", "EKG_TNeg_Klinik", "EKG_LSB_any_Klinik",
            "EchoLVEF", "EchoRWBST", "TropAssay", "InnerklinischeReanimation", "INtensivtherapie", "Todinnerklinisch", "OutcomeSTEMI", "OutcomeNSTEMI", "OutcomeKoronarverschluss",
            "OutcomeInterventionsbedarf", "MACE", "ECG_ERBST", "Troponin_binary"]

orig_cat_cols = list(cat_cols)
cat_cols_use  = [c for c in orig_cat_cols if c in dfm.columns]
missing_cats  = [c for c in orig_cat_cols if c not in dfm.columns]
if missing_cats:
    print(f"Warning: {len(missing_cats)} cat_cols not in dfm and will be ignored:", missing_cats)

# 2) Boolean mask in the order of dfm.columns
cat_mask = dfm.columns.isin(cat_cols_use)
print(f"Categorical features used: {cat_mask.sum()} / {len(cat_mask)}")

# 3) Build cross-sex matrices (uses left_idx/right_idx from Cell 3)
X = dfm.loc[left_idx]
Y = dfm.loc[right_idx]

# 4) Gower distances (rows=X/left, cols=Y/right)
D = gower.gower_matrix(X, Y, cat_features=cat_mask)
D.shape

**Hungarian (optimal one-to-one) matching**

In [ ]:
from scipy.optimize import linear_sum_assignment
caliper = 0.20       # set to None to disable; tune 0.15–0.30 as needed
bigM = 1e6

cost = D.copy()
if caliper is not None:
    cost[cost > caliper] = bigM

nL, nR = cost.shape

# Pad to square for assignment
if nL < nR:
    cost_sq = np.hstack([cost, np.full((nL, nR - nL), bigM)])
elif nL > nR:
    cost_sq = np.vstack([cost, np.full((nL - nR, nR), bigM)])
else:
    cost_sq = cost

row_ind, col_ind = linear_sum_assignment(cost_sq)

# Keep only valid, non-padded, non-caliper-violating matches
valid = [(i, j) for i, j in zip(row_ind, col_ind) if i < nL and j < nR and cost[i, j] < bigM]

**Build the pairs table (no duplicates) + distances**

In [ ]:
if len(valid) == 0:
    pairs = pd.DataFrame(columns=['pair_id','female_id','male_id','gower_distance'])
else:
    left_ids  = X.index.to_numpy()
    right_ids = Y.index.to_numpy()

    tmp = pd.DataFrame(
        [(left_ids[i], right_ids[j], D[i, j]) for i, j in valid],
        columns=['left_id','right_id','gower_distance']
    )

    if left_sex == 'female':
        pairs = tmp.rename(columns={'left_id':'female_id','right_id':'male_id'})
    else:
        pairs = tmp.rename(columns={'left_id':'male_id','right_id':'female_id'})

    pairs.insert(0, 'pair_id', np.arange(1, len(pairs) + 1))

pairs.head()


In [ ]:
# Check uniqueness (no duplicates)
assert pairs['female_id'].is_unique, "Duplicate female IDs found."
assert pairs['male_id'].is_unique,   "Duplicate male IDs found."

# Summary of distances
pairs['gower_distance'].describe()


#### **UMAP visualization**

In [ ]:
import umap

In [ ]:
# Calculate Gower distance matrix
gower_dist = gower.gower_matrix(df_gower, df_gower, cat_features=cat_mask)

In [ ]:
# run UMAP on Gower distance matrix
umap_model = umap.UMAP(metric='precomputed', n_neighbors=12, min_dist=0.3, n_components=2, random_state=42)

# Fit and transform the data
df_umap = umap_model.fit_transform(gower_dist)

# Plot the UMAP embedding
plt.figure(figsize=(8, 8))
plt.scatter(df_umap[:, 0], df_umap[:, 1],  s = 60, c = "#d8d8d8", alpha = 1,
            edgecolor='white', linewidth=0.4)
plt.xlabel('UMAP1')
plt.ylabel('UMAP2')
plt.gca().set_aspect(1./plt.gca().get_data_ratio())
plt.show()

**Plot sex pairs identified by Gower distance**

In [ ]:
index_used = dfm.index
emb_df = pd.DataFrame(df_umap, index=index_used, columns=["UMAP1", "UMAP2"])

# keep only pairs present in the embedding
pairs_use = pairs[
    pairs["female_id"].isin(emb_df.index) & pairs["male_id"].isin(emb_df.index)
].copy()

len(pairs_use), pairs_use.head()

In [ ]:
# Define custom colormap
custom_cmap = ListedColormap(['#c53f60', '#3f478f'])  # 0 = female = red / 1 = male = blue
fig, ax = plt.subplots(figsize=(8,8))
ax.scatter(emb_df["UMAP1"], emb_df["UMAP2"],
           s=60, c=df_imputed["Geschlecht"],  cmap=custom_cmap, alpha=0.8, edgecolor="white", linewidth=0.4, zorder=1)

lc = LineCollection(segs, linewidths=1, alpha=0.15, color = "black", zorder=2)
ax.add_collection(lc)
ax.set_xlabel("UMAP1"); ax.set_ylabel("UMAP2")
ax.set_aspect(1.0/ax.get_data_ratio())
plt.show()

# **Fig. 2: Prehospital twins**

In [ ]:
# list all columns with index
for idx, col in enumerate(df_imputed.columns):
    print(f"Index {idx}: {col}")

In [ ]:
# select columns for Gower Distance patient twin analysis
df_gower = df_imputed.copy()
df_gower = df_gower.iloc[:, [0,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19]] # prehospital twins
print(df_gower.columns)
print("Number of matching variables: ", len(df_gower.columns))

#### **Identify digital twins with opposite sex**

In [ ]:
# Align indices
sex_raw = df_imputed["Geschlecht"].reindex(df_gower.index)

# Map 0/1 to {'female','male'} and drop anything not 0/1
sex = pd.to_numeric(sex_raw, errors='coerce').map({0: 'female', 1: 'male'})
mask_valid = sex.notna()

dfm = df_gower.loc[mask_valid].copy()
sex = sex.loc[mask_valid]
sex.value_counts()

**Split groups & choose which side to fully match**

In [ ]:
fem_idx = dfm.index[sex.eq('female')]
mal_idx = dfm.index[sex.eq('male')]

if len(fem_idx) == 0 or len(mal_idx) == 0:
    raise ValueError("Need at least one female and one male to match.")

# Match everyone in the smaller group (typical)
if len(fem_idx) <= len(mal_idx):
    left_idx, right_idx = fem_idx, mal_idx
    left_sex, right_sex = 'female', 'male'
else:
    left_idx, right_idx = mal_idx, fem_idx
    left_sex, right_sex = 'male', 'female'

**Build Gower distance & compute cross-sex distances**

In [ ]:
# 1) Validate cat_cols against dfm
cat_cols = ["Geschlecht", "EKGSTHebung", "EKGSTSenkungen", "EKGTneg", "EKGLSB_any", "ThoraxschmerzNA", "ThoraxschmerzatypischNA", "DyspnoeNA", "Dyslipidämiebekannt",
            "FAAnamnesebekannt", "Nikotindichbekannt", "Hypertonie", "Diabetes", "BekannteKHK", "GerinnungsmedNA", "BetaBlockerNA", "NitroNA", "AntieemeseNA", "MorphinNA",
            "Thoraxschmerz", "Thoraxschmerzatypisch", "Dyspnoe", "ChronischeNiereninsuffizienz", "EKG_STE_Klinik", "EKG_STD_Klinik", "EKG_TNeg_Klinik", "EKG_LSB_any_Klinik",
            "EchoLVEF", "EchoRWBST", "TropAssay", "InnerklinischeReanimation", "INtensivtherapie", "Todinnerklinisch", "OutcomeSTEMI", "OutcomeNSTEMI", "OutcomeKoronarverschluss",
            "OutcomeInterventionsbedarf", "MACE", "ECG_ERBST", "Troponin_binary"]

orig_cat_cols = list(cat_cols)
cat_cols_use  = [c for c in orig_cat_cols if c in dfm.columns]
missing_cats  = [c for c in orig_cat_cols if c not in dfm.columns]
if missing_cats:
    print(f"Warning: {len(missing_cats)} cat_cols not in dfm and will be ignored:", missing_cats)

# 2) Boolean mask in the order of dfm.columns
cat_mask = dfm.columns.isin(cat_cols_use)
print(f"Categorical features used: {cat_mask.sum()} / {len(cat_mask)}")

# 3) Build cross-sex matrices (uses left_idx/right_idx from Cell 3)
X = dfm.loc[left_idx]
Y = dfm.loc[right_idx]

# 4) Gower distances (rows=X/left, cols=Y/right)
D = gower.gower_matrix(X, Y, cat_features=cat_mask)
D.shape

**Hungarian (optimal one-to-one) matching**

In [ ]:
from scipy.optimize import linear_sum_assignment
caliper = 0.20       # set to None to disable; tune 0.15–0.30 as needed
bigM = 1e6

cost = D.copy()
if caliper is not None:
    cost[cost > caliper] = bigM

nL, nR = cost.shape

# Pad to square for assignment
if nL < nR:
    cost_sq = np.hstack([cost, np.full((nL, nR - nL), bigM)])
elif nL > nR:
    cost_sq = np.vstack([cost, np.full((nL - nR, nR), bigM)])
else:
    cost_sq = cost

row_ind, col_ind = linear_sum_assignment(cost_sq)

# Keep only valid, non-padded, non-caliper-violating matches
valid = [(i, j) for i, j in zip(row_ind, col_ind) if i < nL and j < nR and cost[i, j] < bigM]

**Build the pairs table (no duplicates) + distances**

In [ ]:
if len(valid) == 0:
    pairs = pd.DataFrame(columns=['pair_id','female_id','male_id','gower_distance'])
else:
    left_ids  = X.index.to_numpy()
    right_ids = Y.index.to_numpy()

    tmp = pd.DataFrame(
        [(left_ids[i], right_ids[j], D[i, j]) for i, j in valid],
        columns=['left_id','right_id','gower_distance']
    )

    if left_sex == 'female':
        pairs = tmp.rename(columns={'left_id':'female_id','right_id':'male_id'})
    else:
        pairs = tmp.rename(columns={'left_id':'male_id','right_id':'female_id'})

    pairs.insert(0, 'pair_id', np.arange(1, len(pairs) + 1))

pairs.head()

In [ ]:
# Check uniqueness (no duplicates)
assert pairs['female_id'].is_unique, "Duplicate female IDs found."
assert pairs['male_id'].is_unique,   "Duplicate male IDs found."

# Summary of distances
pairs['gower_distance'].describe()

In [ ]:
# plot histogram with all Gower distances
plt.figure(figsize=(5.5,2))
plt.hist(pairs['gower_distance'], bins=35, color = "#d8d8d8")
plt.axvline(0.05, color = "black", linewidth=0.8, linestyle=(0, (5, 5))) # 10-point dash, 20-point gap
plt.xlabel('Gower distance')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

#### **Create love plot of matching process**

**Keep tight pairs (Gower distance < 0.05) and build the post-match cohort**

In [ ]:
mThreshold = 0.07
vars_match = df_gower.columns
pairs_tight = pairs.loc[pairs["gower_distance"] < mThreshold].copy()

# Make both sides Index → union = unique IDs
ids_f = pd.Index(pairs_tight["female_id"])
ids_m = pd.Index(pairs_tight["male_id"])
keep_ids = ids_f.union(ids_m)

# (robust) ensure IDs exist in dfm and dtype matches
keep_ids = keep_ids.astype(dfm.index.dtype).intersection(dfm.index)

post_X   = dfm.loc[keep_ids, vars_match]
post_sex = sex.loc[keep_ids]

# Pre = full pool used for matching
pre_X, pre_sex = dfm[vars_match], sex

**Compute SMDs (numeric & categorical)**

In [ ]:
# helper functions
def smd_numeric(x, g):
    x1 = x[g=='female']; x2 = x[g=='male']
    v1 = np.nanvar(x1, ddof=1); v2 = np.nanvar(x2, ddof=1)
    sd_p = np.sqrt((v1 + v2)/2.0)
    return (np.nanmean(x1) - np.nanmean(x2)) / (sd_p if sd_p > 0 else np.nan)

def smd_binary_from_indicator(ind, g):
    p1 = np.nanmean(ind[g=='female'])
    p2 = np.nanmean(ind[g=='male'])
    denom = np.sqrt((p1*(1-p1) + p2*(1-p2))/2.0)
    return (p1 - p2) / (denom if denom > 0 else np.nan)

def balance_table(X, g, vars_match, cat_cols):
    rows = []
    for v in vars_match:
        col = X[v]
        if v in cat_cols:
            # multi-level categorical → one dot per level
            levels = pd.Categorical(col).categories if pd.api.types.is_categorical_dtype(col) else pd.Index(sorted(pd.unique(col.dropna())))
            for lvl in levels:
                ind = (col == lvl).astype(float)
                rows.append({"label": f"{v}: {lvl}", "SMD": smd_binary_from_indicator(ind, g)})
        else:
            rows.append({"label": v, "SMD": smd_numeric(col.astype(float), g)})
    out = pd.DataFrame(rows)
    out["absSMD"] = out["SMD"].abs()
    return out.sort_values("absSMD")

In [ ]:
# drop redundant complement indicators like "... : 0.0"
pre_bal  = balance_table(pre_X,  pre_sex,  vars_match, cat_cols).assign(stage="Pre-match")
post_bal = balance_table(post_X, post_sex, vars_match, cat_cols).assign(stage="Post-match (<0.05)")

def drop_zero_levels(df):
    return df[~df["label"].str.endswith(": 0.0", na=False)].copy()

pre_bal  = drop_zero_levels(pre_bal)
post_bal = drop_zero_levels(post_bal)

bal = (pd.concat([pre_bal, post_bal], ignore_index=True)
         .dropna(subset=["SMD"])
         .copy())

# order labels by pre-match |SMD| to make the plot readable
order = (pre_bal
         .dropna(subset=["SMD"])
         .sort_values("absSMD")["label"])
bal["label"] = pd.Categorical(bal["label"], categories=order, ordered=True)

In [ ]:
# Merge pre/post SMDs per label
df_dumb = (pre_bal[['label','SMD']]
           .rename(columns={'SMD':'SMD_pre'})
           .merge(post_bal[['label','SMD']].rename(columns={'SMD':'SMD_post'}),
                  on='label', how='outer'))

# Order by |pre SMD| (fallback to |post| if pre is NaN)
order = (pre_bal.dropna(subset=['SMD'])
                 .sort_values('absSMD')['label'])
df_dumb['label'] = pd.Categorical(df_dumb['label'], categories=order, ordered=True)
df_dumb = df_dumb.sort_values('label').dropna(subset=['label'])

# Plot
plt.figure(figsize=(7, max(4, 0.3*df_dumb.shape[0])))
y = np.arange(df_dumb.shape[0])

for i, r in enumerate(df_dumb.itertuples()):
    x0, x1 = r.SMD_pre, r.SMD_post
    if pd.notna(x0) and pd.notna(x1):
        plt.plot([x0, x1], [i, i], color="#bdbdbd", linewidth=5, alpha = 0.3, zorder=1)   # connector
    if pd.notna(x0):
        plt.scatter(x0, i, c="#888888", s=28, label="Pre-match" if i==0 else None, zorder=2)
    if pd.notna(x1):
        plt.scatter(x1, i, c="#F5A617", s=28, label="Post-match (<0.05)" if i==0 else None, zorder=3)

plt.axvline(0, color="k", linestyle=":")
plt.axvline(0.10, color="#bbbbbb", linestyle="--"); plt.axvline(-0.10, color="#bbbbbb", linestyle="--")
plt.yticks(y, df_dumb['label'])
plt.xlabel("Standardized Mean Difference  (Female − Male)")
plt.legend(loc="lower left", frameon=True)
plt.tight_layout()
plt.show()

#### **Analyze sex differences in outcome**

In [ ]:
df_imputed.columns

In [ ]:
thr = 0.05
pairs_tight = pairs.query("gower_distance < @thr").copy()

# Keep only pairs that exist in df_imputed
pairs_use = pairs_tight[
    pairs_tight.female_id.isin(df_imputed.index) &
    pairs_tight.male_id.isin(df_imputed.index)
].copy()

# define overall general colors
col_f = "#c53f60"  # female
col_m = "#3f478f"  # male

#####**Continuous  variables**

In [ ]:
var_continuous = "CRP" # CRP, Troponinwert, Troponinwert2, hsTroponinwert, hsTroponinwert2, deltaTrop, deltaHsTrop, CK, LDH, Lactat, Hb, Leukozyten, Krea

In [ ]:
sub  = df_imputed.loc[:, [var_continuous, "Geschlecht"]].copy()

# Wide, paired frame
continuous_pairs = pd.DataFrame({
    "pair_id":   pairs_use["pair_id"].to_numpy(),
    "female_id": pairs_use["female_id"].to_numpy(),
    "male_id":   pairs_use["male_id"].to_numpy(),
    "female_value": sub.loc[pairs_use["female_id"], var_continuous].to_numpy(),
    "male_value":   sub.loc[pairs_use["male_id"],   var_continuous].to_numpy(),
})

# Remove extreme outliers
#continuous_pairs = continuous_pairs[(continuous_pairs["female_value"] > 50) & (continuous_pairs["male_value"] > 50)].copy() # outlier in LDH
#continuous_pairs = continuous_pairs[(continuous_pairs["female_value"] > 1000) & (continuous_pairs["male_value"] >1000)].copy()

x_f, x_m = np.zeros(len(continuous_pairs)), np.ones(len(continuous_pairs))

In [ ]:
# Colors + jitter
jitter = 0.03
pos_f, pos_m = -0.15, 1.15
rng = np.random.default_rng(42)

# Clean vectors
f_vals = pd.to_numeric(continuous_pairs["female_value"], errors="coerce").to_numpy()
m_vals = pd.to_numeric(continuous_pairs["male_value"],   errors="coerce").to_numpy()
valid = np.isfinite(f_vals) & np.isfinite(m_vals)
f_clean = f_vals[np.isfinite(f_vals)]
m_clean = m_vals[np.isfinite(m_vals)]

fig, ax = plt.subplots(figsize=(2.5,4))

# Paired connectors with jitter
x_f = rng.normal(0.0, jitter, valid.sum())
x_m = rng.normal(1.0, jitter, valid.sum())
for xf, xm, yf, ym in zip(x_f, x_m, f_vals[valid], m_vals[valid]):
    ax.plot([xf, xm], [yf, ym], color="0.75", linewidth=0.6, alpha=0.3, zorder=1)

# Jittered points
ax.scatter(rng.normal(0.0, jitter, f_clean.size), f_clean,
           c=col_f, s=20, alpha=0.8, label="Female", zorder=2)
ax.scatter(rng.normal(1.0, jitter, m_clean.size), m_clean,
           c=col_m, s=20, alpha=0.8, label="Male",   zorder=2)

# --- Boxplots (draw last) and push zorder high so they're on top ---
bp_f = ax.boxplot([f_clean], positions=[0], widths=0.12, vert=True, boxprops=dict(color="black", facecolor="white"), medianprops=dict(color="black"),capprops=dict(visible=False),
                  patch_artist=True, showfliers=False)
bp_m = ax.boxplot([m_clean], positions=[1], widths=0.12, vert=True, boxprops=dict(color="black", facecolor="white"), medianprops=dict(color="black"),capprops=dict(visible=False),
                  patch_artist=True, showfliers=False)

#ax.set_xticks([0, 1]); ax.set_xticklabels(["Female", "Male"])
ax.tick_params(axis='x', which='both', bottom=False, top=False, labelbottom=False)
ax.set_xlim(pos_f - 0.4, pos_m + 0.4)
ax.set_ylabel(f"{var_continuous} [mg/dl]")
ax.set_yscale("log")
plt.tight_layout()
plt.show()

In [ ]:
from scipy import stats

f = pd.to_numeric(continuous_pairs["female_value"], errors="coerce")
m = pd.to_numeric(continuous_pairs["male_value"],   errors="coerce")

mask = f.notna() & m.notna()
diff = (f[mask] - m[mask]).to_numpy()
n = diff.size

# Paired t-test
tres = stats.ttest_rel(f[mask], m[mask])
# Wilcoxon (nonparametric)
wres = stats.wilcoxon(f[mask], m[mask], zero_method="wilcox", alternative="two-sided")

# Mean diff CI
se = diff.std(ddof=1) / np.sqrt(n)
tcrit = stats.t.ppf(0.975, n-1)
mean_diff = diff.mean()
ci = (mean_diff - tcrit*se, mean_diff + tcrit*se)

# Paired Cohen's dz
cohen_dz = mean_diff / diff.std(ddof=1)

print(f"Pairs: {n}")
print(f"Mean difference (Female − Male): {mean_diff:.2g}  (95% CI {ci[0]:.2g} to {ci[1]:.2g})")
print(f"Wilcoxon signed-rank p = {wres.pvalue:.3g}")
print(f"Cohen's dz = {cohen_dz:.3g}")

#####**Categorical variables**

In [ ]:
var_categorical = "Todinnerklinisch"  # InnerklinischeReanimation, INtensivtherapie, Todinnerklinisch, OutcomeSTEMI, OutcomeNSTEMI, OutcomeKoronarverschluss, OutcomeInterventionsbedarf, MACE

In [ ]:
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.proportion import proportion_confint

sub = df_imputed.loc[:, [var_categorical, "Geschlecht"]].copy()
category_pairs = pd.DataFrame({
    "pair_id":     pairs_use["pair_id"].to_numpy(),
    "female_id":   pairs_use["female_id"].to_numpy(),
    "male_id":     pairs_use["male_id"].to_numpy(),
    "female_out":  sub.loc[pairs_use["female_id"], var_categorical].to_numpy(),
    "male_out":    sub.loc[pairs_use["male_id"],   var_categorical].to_numpy(),
})

# --- Clean to 0/1; drop pairs with missing outcome on either side ---
fo_raw = pd.to_numeric(pd.Series(category_pairs["female_out"]), errors="coerce")
mo_raw = pd.to_numeric(pd.Series(category_pairs["male_out"]),   errors="coerce")
mask = fo_raw.notna() & mo_raw.notna()
f_o = (fo_raw[mask] > 0).astype(int)   # positive -> 1, else -> 0
m_o = (mo_raw[mask] > 0).astype(int)

rate_f, rate_m = f_o.mean()*100, m_o.mean()*100

# --- Plot ---
plt.figure(figsize=(2.5,4))
plt.bar(["Female","Male"], [rate_f, rate_m], color=[col_f, col_m])
plt.ylabel(f"{var_categorical} (%)")
plt.xticks([])
plt.tight_layout()
plt.show()

In [ ]:
# 2x2 table and McNemar test
tab = pd.crosstab(f_o, m_o).reindex(index=[0,1], columns=[0,1], fill_value=0)

# Cells
a = int(tab.loc[0,0])
b = int(tab.loc[0,1])  # Male yes / Female no
c = int(tab.loc[1,0])  # Female yes / Male no
d = int(tab.loc[1,1])

# Totals
D = b + c                               # discordant pairs
N = int(tab.to_numpy().sum())           # total non-missing paired observations
# (equivalently: N = a + b + c + d)

mc = mcnemar(tab, exact=(D <= 25))
p_mc = mc.pvalue

# Matched OR (Female/Male) via discordant pairs
if D > 0:
    p_hat = c / D
    ci_low_p, ci_upp_p = proportion_confint(c, D, alpha=0.05, method="beta")
    or_hat = (p_hat / (1 - p_hat)) if 0 < p_hat < 1 else np.inf
    or_ci  = (ci_low_p/(1-ci_low_p), ci_upp_p/(1-ci_upp_p))
else:
    or_hat, or_ci = np.nan, (np.nan, np.nan)

# Absolute risk difference (Female - Male)
rd = (c - b) / N

# 95% CI via transforming the exact CI on p = c / (b+c):
if D > 0:
    rd_ci_low = (2*ci_low_p - 1) * (D / N)
    rd_ci_upp = (2*ci_upp_p - 1) * (D / N)
else:
    rd_ci_low = rd_ci_upp = np.nan

print("2x2 (Female vs Male):\n", tab)
print(f"Pairs analyzed (complete): {N}  |  Concordant a={a}, d={d}  |  Discordant b={b}, c={c} (D={D})")
print(f"McNemar p = {p_mc:.4g}")
print(f"Matched OR (Female/Male)≈{or_hat:.3g}  95%CI [{or_ci[0]:.3g}, {or_ci[1]:.3g}]")
print(f"Absolute risk difference (Female − Male): {rd*100:.1f} pp "
      f"95%CI [{rd_ci_low*100:.1f}, {rd_ci_upp*100:.1f} pp]")


#####**Ordinal variables**

In [ ]:
var_ordinal = "EchoLVEF"

In [ ]:
df_imputed['EchoLVEF'].value_counts()

In [ ]:
from matplotlib.patches import Patch

# Pull from full DB for just the matched IDs you kept (pairs_use)
sub = df_imputed.loc[:, [var_ordinal, "Geschlecht"]].copy()

ord_pairs = pd.DataFrame({
    "pair_id":    pairs_use["pair_id"].to_numpy(),
    "female_id":  pairs_use["female_id"].to_numpy(),
    "male_id":    pairs_use["male_id"].to_numpy(),
    "female_val": sub.loc[pairs_use["female_id"], var_ordinal].to_numpy(),
    "male_val":   sub.loc[pairs_use["male_id"],   var_ordinal].to_numpy(),
})

# Clean to integers 1..4 and drop pairs with missing
f_raw = pd.to_numeric(pd.Series(ord_pairs["female_val"]), errors="coerce")
m_raw = pd.to_numeric(pd.Series(ord_pairs["male_val"]),   errors="coerce")
mask = f_raw.notna() & m_raw.notna()
f = f_raw[mask].astype(int).clip(0, 3)
m = m_raw[mask].astype(int).clip(0, 3)

cats = [0, 1, 2, 3]
n = len(f)

# proportions per category
f_counts = f.value_counts().reindex(cats, fill_value=0)
m_counts = m.value_counts().reindex(cats, fill_value=0)
f_pct = (f_counts / n * 100).to_numpy()
m_pct = (m_counts / n * 100).to_numpy()

# transparency per category (light→dark for 4→1)
alphas = [1, 0.75, 0.5, 0.4] # LVEF=4 highest transparency=lightest, LVEF=1 is darkest

fig, ax = plt.subplots(figsize=(2.5, 4))

# stack bars
bottom_f = 0.0
bottom_m = 0.0
for i, (pF, pM) in enumerate(zip(f_pct, m_pct), start=1):
    ax.bar(0, pF, bottom=bottom_f, color=col_f, alpha=alphas[i-1], width=0.6, edgecolor='white', linewidth=0.6)
    ax.bar(1, pM, bottom=bottom_m, color=col_m, alpha=alphas[i-1], width=0.6, edgecolor='white', linewidth=0.6)
    bottom_f += pF
    bottom_m += pM
sex_handles = [Patch(facecolor=col_f, label="Female"),
               Patch(facecolor=col_m, label="Male")]


# plot
ax.set_xticks([0, 1]); ax.set_xticklabels(["Female", "Male"])
ax.set_ylabel(f"{var_ordinal} distribution (%)")
ax.set_ylim(0, 100)
plt.xticks([])
plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import wilcoxon

# Treat 1<2<3<4 as ordered scale; Wilcoxon signed-rank on coded levels
w = wilcoxon(f.to_numpy(), m.to_numpy(), zero_method="wilcox", alternative="two-sided")
print(f"Wilcoxon signed-rank (ordinal, Female − Male): p = {w.pvalue:.4g}  (n={n} pairs)")

# Shift table: Female−Male category differences
diff = f.to_numpy() - m.to_numpy()
shift = pd.Series(diff).value_counts().sort_index()
print("\nShift table (Female − Male category):")
print(shift.to_string())

# Also show marginal distributions (counts)
print("\nCounts per category (Female):")
print(f_counts.to_string())
print("\nCounts per category (Male):")
print(m_counts.to_string())

# show mean category difference
diff = (f - m).to_numpy()
delta = diff.mean()
rng = np.random.default_rng(42)
boot = np.array([rng.choice(diff, size=diff.size, replace=True).mean() for _ in range(10000)]) # 10,000x bootstrapping
ci_lo, ci_hi = np.quantile(boot, [0.025, 0.975])
print(f"\nMean category shift (Female−Male): {delta:.3f}  (95% CI {ci_lo:.3f} to {ci_hi:.3f})")

#### **Rev1: Create table with results (to check different caliper thresholds)**

In [ ]:
# define all thresholds and variables of interest
thresholds = [0.03, 0.05, 0.07]

continuous_vars = [
    "Troponinwert", "Troponinwert2", "hsTroponinwert", "hsTroponinwert2",
    "deltaTrop", "deltaHsTrop", "CK", "LDH", "Lactat", "Hb", "Leukozyten", "Krea", "CRP"
]

categorical_vars = [
    "InnerklinischeReanimation", "INtensivtherapie", "Todinnerklinisch",
    "OutcomeSTEMI", "OutcomeNSTEMI", "OutcomeKoronarverschluss",
    "OutcomeInterventionsbedarf", "MACE"
]

ordinal_vars = ["EchoLVEF"]

# keep only columns that actually exist
continuous_vars = [v for v in continuous_vars if v in df_imputed.columns]
categorical_vars = [v for v in categorical_vars if v in df_imputed.columns]
ordinal_vars    = [v for v in ordinal_vars    if v in df_imputed.columns]


# helper: get matched pairs for a single threshold
def get_pairs_use(pairs, df_source, thr):
    pairs_tight = pairs.query("gower_distance < @thr").copy()
    pairs_use = pairs_tight[
        pairs_tight.female_id.isin(df_source.index) &
        pairs_tight.male_id.isin(df_source.index)
    ].copy()
    return pairs_use

##### Continuous variable sensitivity table

In [ ]:
def summarize_continuous_var(pairs, df_source, var, thr):
    pairs_use = get_pairs_use(pairs, df_source, thr)

    sub = df_source.loc[:, [var]].copy()
    dat = pd.DataFrame({
        "female": sub.loc[pairs_use["female_id"], var].to_numpy(),
        "male":   sub.loc[pairs_use["male_id"],   var].to_numpy(),
    })

    f = pd.to_numeric(dat["female"], errors="coerce")
    m = pd.to_numeric(dat["male"],   errors="coerce")
    mask = f.notna() & m.notna()
    f, m = f[mask], m[mask]

    if len(f) == 0:
        return {
            "threshold": thr, "variable": var, "type": "continuous",
            "pairs_total": len(pairs_use), "pairs_analyzed": 0,
            "p_value": np.nan, "delta": np.nan, "OR": np.nan
        }

    diff = (f - m).to_numpy()
    wres = stats.wilcoxon(f, m, zero_method="wilcox", alternative="two-sided")

    return {
        "threshold": thr, "variable": var, "type": "continuous",
        "pairs_total": len(pairs_use), "pairs_analyzed": len(f),
        "p_value": wres.pvalue,
        "delta": diff.mean(),
        "OR": np.nan
    }

##### Categorical variable sensitivity table

In [ ]:
from statsmodels.stats.contingency_tables import mcnemar

def summarize_categorical_var(pairs, df_source, var, thr):
    pairs_use = get_pairs_use(pairs, df_source, thr)

    sub = df_source.loc[:, [var]].copy()
    dat = pd.DataFrame({
        "female": sub.loc[pairs_use["female_id"], var].to_numpy(),
        "male":   sub.loc[pairs_use["male_id"],   var].to_numpy(),
    })

    f_raw = pd.to_numeric(dat["female"], errors="coerce")
    m_raw = pd.to_numeric(dat["male"],   errors="coerce")
    mask = f_raw.notna() & m_raw.notna()

    f = (f_raw[mask] > 0).astype(int)
    m = (m_raw[mask] > 0).astype(int)

    if len(f) == 0:
        return {
            "threshold": thr, "variable": var, "type": "categorical",
            "pairs_total": len(pairs_use), "pairs_analyzed": 0,
            "p_value": np.nan, "delta": np.nan, "OR": np.nan
        }

    tab = pd.crosstab(f, m).reindex(index=[0,1], columns=[0,1], fill_value=0)
    b = int(tab.loc[0,1])  # male only
    c = int(tab.loc[1,0])  # female only
    D = b + c

    mc = mcnemar(tab, exact=(D <= 25))

    if D > 0:
        p_hat = c / D
        OR = (p_hat / (1 - p_hat)) if 0 < p_hat < 1 else np.inf
    else:
        OR = np.nan

    return {
        "threshold": thr, "variable": var, "type": "categorical",
        "pairs_total": len(pairs_use), "pairs_analyzed": int(tab.to_numpy().sum()),
        "p_value": mc.pvalue,
        "delta": np.nan,
        "OR": OR
    }

##### Ordinal variable sensitivity table

In [ ]:
from scipy.stats import wilcoxon

def summarize_ordinal_var(pairs, df_source, var, thr):
    pairs_use = get_pairs_use(pairs, df_source, thr)

    sub = df_source.loc[:, [var]].copy()
    dat = pd.DataFrame({
        "female": sub.loc[pairs_use["female_id"], var].to_numpy(),
        "male":   sub.loc[pairs_use["male_id"],   var].to_numpy(),
    })

    f_raw = pd.to_numeric(dat["female"], errors="coerce")
    m_raw = pd.to_numeric(dat["male"],   errors="coerce")
    mask = f_raw.notna() & m_raw.notna()

    f = f_raw[mask].astype(int)
    m = m_raw[mask].astype(int)

    if len(f) == 0:
        return {
            "threshold": thr, "variable": var, "type": "ordinal",
            "pairs_total": len(pairs_use), "pairs_analyzed": 0,
            "p_value": np.nan, "delta": np.nan, "OR": np.nan
        }

    w = wilcoxon(f.to_numpy(), m.to_numpy(), zero_method="wilcox", alternative="two-sided")
    delta = (f - m).mean()

    return {
        "threshold": thr, "variable": var, "type": "ordinal",
        "pairs_total": len(pairs_use), "pairs_analyzed": len(f),
        "p_value": w.pvalue,
        "delta": delta,
        "OR": np.nan
    }

##### Merge variables into one table

In [ ]:
rows = []

for thr in thresholds:
    for var in continuous_vars:
        rows.append(summarize_continuous_var(pairs, df_imputed, var, thr))
    for var in categorical_vars:
        rows.append(summarize_categorical_var(pairs, df_imputed, var, thr))
    for var in ordinal_vars:
        rows.append(summarize_ordinal_var(pairs, df_imputed, var, thr))

sens_all_long = pd.DataFrame(rows)
sens_all_long

In [ ]:
# one wide table for better comparability
sens_all_wide = sens_all_long.pivot(
    index=["type", "variable"],
    columns="threshold",
    values=["pairs_total", "pairs_analyzed", "p_value", "delta", "OR"]
)

sens_all_wide.columns = [f"{metric}_thr{thr}" for metric, thr in sens_all_wide.columns]
sens_all_wide = sens_all_wide.reset_index()

sens_all_wide

In [ ]:
# round
sens_all_table = sens_all_wide.copy()

for c in sens_all_table.columns:
    if c.startswith("p_value_"):
        sens_all_table[c] = sens_all_table[c].map(lambda x: f"{x:.3g}" if pd.notna(x) else "")
    elif c.startswith("delta_"):
        sens_all_table[c] = sens_all_table[c].map(lambda x: f"{x:.2f}" if pd.notna(x) else "")
    elif c.startswith("OR_"):
        sens_all_table[c] = sens_all_table[c].map(
            lambda x: ("∞" if np.isinf(x) else f"{x:.2f}") if pd.notna(x) else ""
        )
    elif c.startswith("pairs_"):
        sens_all_table[c] = sens_all_table[c].map(lambda x: int(x) if pd.notna(x) else "")

sens_all_table

##### Display SMD results in one table

In [ ]:
def summarize_smd_threshold(pairs, dfm, sex, vars_match, cat_cols, thr):
    pairs_use = get_pairs_use(pairs, dfm, thr)

    ids_f = pd.Index(pairs_use["female_id"])
    ids_m = pd.Index(pairs_use["male_id"])
    keep_ids = ids_f.union(ids_m).astype(dfm.index.dtype).intersection(dfm.index)

    post_X   = dfm.loc[keep_ids, vars_match]
    post_sex = sex.loc[keep_ids]

    post_bal = balance_table(post_X, post_sex, vars_match, cat_cols).copy()
    post_bal = drop_zero_levels(post_bal)
    post_bal = post_bal.dropna(subset=["SMD"])

    return {
        "threshold": thr,
        "pairs_total": len(pairs_use),
        "mean_abs_SMD": post_bal["absSMD"].mean(),
        "median_abs_SMD": post_bal["absSMD"].median(),
        "max_abs_SMD": post_bal["absSMD"].max(),
        "n_covariates_gt_0.1": int((post_bal["absSMD"] > 0.1).sum())
    }

smd_rows = [
    summarize_smd_threshold(pairs, dfm, sex, vars_match, cat_cols, thr)
    for thr in thresholds
]

smd_table = pd.DataFrame(smd_rows)
smd_table

In [ ]:
smd_table_fmt = smd_table.copy()
for c in ["mean_abs_SMD", "median_abs_SMD", "max_abs_SMD"]:
    smd_table_fmt[c] = smd_table_fmt[c].map(lambda x: f"{x:.3f}" if pd.notna(x) else "")
smd_table_fmt

# **Fig. 3: Subgroup analysis prehospital treatment**

In [ ]:
# add new variables based on GerinnungsmedNA (0=kein 1=ASS 2=Heparin 3=beides 4=andere)
df_imputed['Gerinnungsmed_keine'] = (df_imputed['GerinnungsmedNA'] == 0).astype(float)
df_imputed['Gerinnungsmed_ASS'] = (df_imputed['GerinnungsmedNA'] == 1).astype(float)
df_imputed['Gerinnungsmed_Heparin'] = (df_imputed['GerinnungsmedNA'] == 2).astype(float)
df_imputed['Gerinnungsmed_ASS_Heparin'] = (df_imputed['GerinnungsmedNA'] == 3).astype(float)

In [ ]:
# prehospital treatment vars
treat_vars  = ['BetaBlockerNA', 'NitroNA', 'AntieemeseNA', 'MorphinNA', 'Gerinnungsmed_keine', 'Gerinnungsmed_ASS', 'Gerinnungsmed_Heparin', 'Gerinnungsmed_ASS_Heparin']
treat_vars = [v for v in treat_vars if v in df_imputed.columns]  # keep existing only

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.proportion import proportion_confint

def summarize_matched_or(df, pairs, var):
    # Pull aligned arrays in the SAME pair order (no index-alignment bugs)
    f_raw = df[var].reindex(pairs["female_id"]).astype(float).to_numpy()
    m_raw = df[var].reindex(pairs["male_id"]).astype(float).to_numpy()

    # Complete-case mask by position
    mask = np.isfinite(f_raw) & np.isfinite(m_raw)
    f = (f_raw[mask] > 0).astype(int)
    m = (m_raw[mask] > 0).astype(int)

    N = f.size
    if N == 0:
        return {"variable": var, "N_pairs": 0, "b_Monly": 0, "c_Fonly": 0, "discordant": 0,
                "rate_F": np.nan, "rate_M": np.nan,
                "OR_hat": np.nan, "OR_lo": np.nan, "OR_hi": np.nan,
                "RD": np.nan, "RD_lo": np.nan, "RD_hi": np.nan, "p_mcnemar": np.nan}

    # 2x2 counts on paired vectors
    a = int(((f==0) & (m==0)).sum())
    b = int(((f==0) & (m==1)).sum())   # Male yes / Female no
    c = int(((f==1) & (m==0)).sum())   # Female yes / Male no
    d = int(((f==1) & (m==1)).sum())
    D = b + c

    # Event rates
    rate_F = float(f.mean()); rate_M = float(m.mean())

    # McNemar test
    tab = np.array([[a, b], [c, d]])
    p_mc = mcnemar(tab, exact=(D <= 25)).pvalue

    # Discordant-pairs OR + exact CI via Beta on p = c/(b+c)
    if D > 0:
        p_hat = c / D
        lo_p, hi_p = proportion_confint(c, D, alpha=0.05, method="beta")
        OR_hat = (p_hat/(1-p_hat)) if 0 < p_hat < 1 else (np.inf if p_hat == 1 else 0.0)
        OR_lo  = 0.0 if lo_p == 0 else (np.inf if lo_p == 1 else lo_p/(1-lo_p))
        OR_hi  = 0.0 if hi_p == 0 else (np.inf if hi_p == 1 else hi_p/(1-hi_p))
        RD     = (c - b) / N
        RD_lo  = (2*lo_p - 1) * (D / N)
        RD_hi  = (2*hi_p - 1) * (D / N)
    else:
        OR_hat = OR_lo = OR_hi = np.nan
        RD = RD_lo = RD_hi = np.nan

    return {"variable": var, "N_pairs": N, "b_Monly": b, "c_Fonly": c, "discordant": D,
            "rate_F": rate_F, "rate_M": rate_M,
            "OR_hat": OR_hat, "OR_lo": OR_lo, "OR_hi": OR_hi,
            "RD": RD, "RD_lo": RD_lo, "RD_hi": RD_hi, "p_mcnemar": p_mc}

##### **Forest plot for all**

In [ ]:
summary = pd.DataFrame([summarize_matched_or(df_imputed, pairs_use, v) for v in treat_vars])
summary = summary.iloc[::-1].reset_index(drop=True)
summary

In [ ]:
dfp = summary.copy()
xmin, xmax = -0.1, 2.1
y = np.arange(len(dfp))

# create forest plot
fig, ax = plt.subplots(figsize=(6.5, 3.5))

lo = dfp["OR_lo"].replace([0, np.inf], [xmin, xmax]).to_numpy()
hi = dfp["OR_hi"].replace([0, np.inf], [xmin, xmax]).to_numpy()
pt = dfp["OR_hat"].to_numpy()

# CIs
for i in range(len(dfp)):
    ax.plot([lo[i], hi[i]], [y[i], y[i]], color="black", linewidth=2, zorder=2)
# Points
ax.scatter(pt, y, s=35, color="black", zorder=2)

ax.axvline(1.0, color="#bbbbbb", linestyle="--", linewidth=1, zorder=1)
ax.set_xlim(xmin, xmax)
ax.set_yticks(y); ax.set_yticklabels(dfp["variable"]); ax.yaxis.set_ticks_position('both')
ax.set_xlabel("Matched odds ratio (female / male)")
#ax.set_title("Treatment differences within matched pairs (95% CI)")

# Annotations
for i, r in dfp.iterrows():
    def fmt_or(x):
        if np.isnan(x): return "NA"
        if np.isinf(x): return "∞"
        return f"{x:.2f}"
    ax.text(xmax*1.07, y[i]-0.04,
            #f" OR={or_txt} [{ci_l}-{ci_h}], P={r['p_mcnemar']:.3g}",
            f"OR={fmt_or(r['OR_hat'])} [{fmt_or(r['OR_lo'])}-{fmt_or(r['OR_hi'])}], P={r['p_mcnemar']:.3g} ",
            va="center", fontsize=9)
ax.margins(y=0.10)
plt.subplots_adjust(right=0.72)
plt.tight_layout()
plt.show()

##### **Forest plot for outcome+ subcohort**

In [ ]:
df_outcome = df_imputed[df_imputed['Troponin_binary'] == 0] # # InnerklinischeReanimation, OutcomeSTEMI, OutcomeNSTEMI, OutcomeKoronarverschluss, OutcomeInterventionsbedarf, MACE, Troponin_binary
len(df_outcome)

# keep only pairs where BOTH members are in the STEMI subcohort
pairs_use_outcome = (
    pairs_use[
        pairs_use["female_id"].isin(df_outcome.index) &
        pairs_use["male_id"].isin(df_outcome.index)
    ]
    .reset_index(drop=True)
)

print(f"Pairs kept: {len(pairs_use_outcome)} / {len(pairs_use)}")

In [ ]:
summary_outcome = pd.DataFrame([summarize_matched_or(df_outcome, pairs_use_outcome, v) for v in treat_vars])
summary_outcome = summary_outcome.iloc[::-1].reset_index(drop=True)
summary_outcome

In [ ]:
dfp = summary_outcome.copy()
xmin, xmax = -0.1, 2.1
y = np.arange(len(dfp))

# create forest plot
fig, ax = plt.subplots(figsize=(6.5, 3.5))

lo = dfp["OR_lo"].replace([0, np.inf], [xmin, xmax]).to_numpy()
hi = dfp["OR_hi"].replace([0, np.inf], [xmin, xmax]).to_numpy()
pt = dfp["OR_hat"].to_numpy()

# CIs
for i in range(len(dfp)):
    ax.plot([lo[i], hi[i]], [y[i], y[i]], color="black", linewidth=2, zorder=2)
# Points
ax.scatter(pt, y, s=35, color="black", zorder=2)

ax.axvline(1.0, color="#bbbbbb", linestyle="--", linewidth=1, zorder=1)
ax.set_xlim(xmin, xmax)
ax.set_yticks(y); ax.set_yticklabels(dfp["variable"]); ax.yaxis.set_ticks_position('both')
ax.set_xlabel("Matched odds ratio (female / male)")

# Annotations
for i, r in dfp.iterrows():
    def fmt_or(x):
        if np.isnan(x): return "NA"
        if np.isinf(x): return "∞"
        return f"{x:.2f}"
    ax.text(xmax*1.07, y[i]-0.04,
            f"OR={fmt_or(r['OR_hat'])} [{fmt_or(r['OR_lo'])}-{fmt_or(r['OR_hi'])}], P={r['p_mcnemar']:.3g} ",
            va="center", fontsize=9)
ax.margins(y=0.10)
plt.subplots_adjust(right=0.72)
plt.tight_layout()
plt.show()

# **Fig. 4: In-hospital twins**

In [ ]:
# list all columns with index
for idx, col in enumerate(df_imputed.columns):
    print(f"Index {idx}: {col}")

In [ ]:
# select columns for Gower Distance patient twin analysis
df_gower = df_imputed.copy()
df_gower = df_gower.iloc[:, [29,30,31,32,33, 34, 58, 40, 41, 42, 47,48,49,50,51,52,53,54]] # in-hospital twins
print(df_gower.columns)
print("Number of matching variables: ", len(df_gower.columns))

#### **Identify digital twins with opposite sex**

In [ ]:
# Align indices
sex_raw = df_imputed["Geschlecht"].reindex(df_gower.index)

# Map 0/1 to {'female','male'} and drop anything not 0/1
sex = pd.to_numeric(sex_raw, errors='coerce').map({0: 'female', 1: 'male'})
mask_valid = sex.notna()

dfm = df_gower.loc[mask_valid].copy()
sex = sex.loc[mask_valid]
sex.value_counts()

**Split groups & choose which side to fully match**

In [ ]:
fem_idx = dfm.index[sex.eq('female')]
mal_idx = dfm.index[sex.eq('male')]

if len(fem_idx) == 0 or len(mal_idx) == 0:
    raise ValueError("Need at least one female and one male to match.")

# Match everyone in the smaller group (typical)
if len(fem_idx) <= len(mal_idx):
    left_idx, right_idx = fem_idx, mal_idx
    left_sex, right_sex = 'female', 'male'
else:
    left_idx, right_idx = mal_idx, fem_idx
    left_sex, right_sex = 'male', 'female'

**Build Gower distance & compute cross-sex distances**

In [ ]:
# 1) Validate cat_cols against dfm
cat_cols = ["Geschlecht", "EKGSTHebung", "EKGSTSenkungen", "EKGTneg", "EKGLSB_any", "ThoraxschmerzNA", "ThoraxschmerzatypischNA", "DyspnoeNA", "Dyslipidämiebekannt",
            "FAAnamnesebekannt", "Nikotindichbekannt", "Hypertonie", "Diabetes", "BekannteKHK", "GerinnungsmedNA", "BetaBlockerNA", "NitroNA", "AntieemeseNA", "MorphinNA",
            "Thoraxschmerz", "Thoraxschmerzatypisch", "Dyspnoe", "ChronischeNiereninsuffizienz", "EKG_STE_Klinik", "EKG_STD_Klinik", "EKG_TNeg_Klinik", "EKG_LSB_any_Klinik",
            "EchoLVEF", "EchoRWBST", "TropAssay", "InnerklinischeReanimation", "INtensivtherapie", "Todinnerklinisch", "OutcomeSTEMI", "OutcomeNSTEMI", "OutcomeKoronarverschluss",
            "OutcomeInterventionsbedarf", "MACE", "ECG_ERBST", "Troponin_binary"]

orig_cat_cols = list(cat_cols)
cat_cols_use  = [c for c in orig_cat_cols if c in dfm.columns]
missing_cats  = [c for c in orig_cat_cols if c not in dfm.columns]
if missing_cats:
    print(f"Warning: {len(missing_cats)} cat_cols not in dfm and will be ignored:", missing_cats)

# 2) Boolean mask in the order of dfm.columns
cat_mask = dfm.columns.isin(cat_cols_use)
print(f"Categorical features used: {cat_mask.sum()} / {len(cat_mask)}")

# 3) Build cross-sex matrices (uses left_idx/right_idx from Cell 3)
X = dfm.loc[left_idx]
Y = dfm.loc[right_idx]

# 4) Gower distances (rows=X/left, cols=Y/right)
D = gower.gower_matrix(X, Y, cat_features=cat_mask)
D.shape

**Hungarian (optimal one-to-one) matching with optional caliper**

In [ ]:
from scipy.optimize import linear_sum_assignment
caliper = 0.20       # set to None to disable; tune 0.15–0.30 as needed
bigM = 1e6

cost = D.copy()
if caliper is not None:
    cost[cost > caliper] = bigM

nL, nR = cost.shape

# Pad to square for assignment
if nL < nR:
    cost_sq = np.hstack([cost, np.full((nL, nR - nL), bigM)])
elif nL > nR:
    cost_sq = np.vstack([cost, np.full((nL - nR, nR), bigM)])
else:
    cost_sq = cost

row_ind, col_ind = linear_sum_assignment(cost_sq)

# Keep only valid, non-padded, non-caliper-violating matches
valid = [(i, j) for i, j in zip(row_ind, col_ind) if i < nL and j < nR and cost[i, j] < bigM]

**Build the pairs table (no duplicates) + distances**

In [ ]:
if len(valid) == 0:
    pairs = pd.DataFrame(columns=['pair_id','female_id','male_id','gower_distance'])
else:
    left_ids  = X.index.to_numpy()
    right_ids = Y.index.to_numpy()

    tmp = pd.DataFrame(
        [(left_ids[i], right_ids[j], D[i, j]) for i, j in valid],
        columns=['left_id','right_id','gower_distance']
    )

    if left_sex == 'female':
        pairs = tmp.rename(columns={'left_id':'female_id','right_id':'male_id'})
    else:
        pairs = tmp.rename(columns={'left_id':'male_id','right_id':'female_id'})

    pairs.insert(0, 'pair_id', np.arange(1, len(pairs) + 1))

pairs.head()

In [ ]:
# Check uniqueness (no duplicates)
assert pairs['female_id'].is_unique, "Duplicate female IDs found."
assert pairs['male_id'].is_unique,   "Duplicate male IDs found."

# Summary of distances
pairs['gower_distance'].describe()

In [ ]:
# plot histogram with all Gower distances
plt.figure(figsize=(5.5,2))
plt.hist(pairs['gower_distance'], bins=50, color = "#d8d8d8")
plt.axvline(0.025, color = "black", linewidth=0.8, linestyle=(0, (5, 5))) # 10-point dash, 20-point gap
plt.xlabel('Gower distance')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# get number of pairs with gower distance <0.025
len(pairs.query("gower_distance < 0.025"))

#### **Create love plot of matching process**

**Keep tight pairs (Gower distance < 0.05) and build the post-match cohort**

In [ ]:
mThreshold = 0.025 # Rev1: changed this from 0.05 to 0.025
vars_match = df_gower.columns
pairs_tight = pairs.loc[pairs["gower_distance"] < mThreshold].copy()

# Make both sides Index → union = unique IDs
ids_f = pd.Index(pairs_tight["female_id"])
ids_m = pd.Index(pairs_tight["male_id"])
keep_ids = ids_f.union(ids_m)

# (robust) ensure IDs exist in dfm and dtype matches
keep_ids = keep_ids.astype(dfm.index.dtype).intersection(dfm.index)

post_X   = dfm.loc[keep_ids, vars_match]
post_sex = sex.loc[keep_ids]

# Pre = full pool used for matching
pre_X, pre_sex = dfm[vars_match], sex

**Compute SMDs (numeric & categorical)**

In [ ]:
# helper functions
def smd_numeric(x, g):
    x1 = x[g=='female']; x2 = x[g=='male']
    v1 = np.nanvar(x1, ddof=1); v2 = np.nanvar(x2, ddof=1)
    sd_p = np.sqrt((v1 + v2)/2.0)
    return (np.nanmean(x1) - np.nanmean(x2)) / (sd_p if sd_p > 0 else np.nan)

def smd_binary_from_indicator(ind, g):
    p1 = np.nanmean(ind[g=='female'])
    p2 = np.nanmean(ind[g=='male'])
    denom = np.sqrt((p1*(1-p1) + p2*(1-p2))/2.0)
    return (p1 - p2) / (denom if denom > 0 else np.nan)

def balance_table(X, g, vars_match, cat_cols):
    rows = []
    for v in vars_match:
        col = X[v]
        if v in cat_cols:
            # multi-level categorical → one dot per level
            levels = pd.Categorical(col).categories if pd.api.types.is_categorical_dtype(col) else pd.Index(sorted(pd.unique(col.dropna())))
            for lvl in levels:
                ind = (col == lvl).astype(float)
                rows.append({"label": f"{v}: {lvl}", "SMD": smd_binary_from_indicator(ind, g)})
        else:
            rows.append({"label": v, "SMD": smd_numeric(col.astype(float), g)})
    out = pd.DataFrame(rows)
    out["absSMD"] = out["SMD"].abs()
    return out.sort_values("absSMD")

In [ ]:
# drop redundant complement indicators like "... : 0.0"
pre_bal  = balance_table(pre_X,  pre_sex,  vars_match, cat_cols).assign(stage="Pre-match")
post_bal = balance_table(post_X, post_sex, vars_match, cat_cols).assign(stage="Post-match (<0.05)")

def drop_zero_levels(df):
    return df[~df["label"].str.endswith(": 0.0", na=False)].copy()

pre_bal  = drop_zero_levels(pre_bal)
post_bal = drop_zero_levels(post_bal)

bal = (pd.concat([pre_bal, post_bal], ignore_index=True)
         .dropna(subset=["SMD"])
         .copy())

# order labels by pre-match |SMD| to make the plot readable
order = (pre_bal
         .dropna(subset=["SMD"])
         .sort_values("absSMD")["label"])
bal["label"] = pd.Categorical(bal["label"], categories=order, ordered=True)

In [ ]:
# Merge pre/post SMDs per label
df_dumb = (pre_bal[['label','SMD']]
           .rename(columns={'SMD':'SMD_pre'})
           .merge(post_bal[['label','SMD']].rename(columns={'SMD':'SMD_post'}),
                  on='label', how='outer'))

# Order by |pre SMD| (fallback to |post| if pre is NaN)
order = (pre_bal.dropna(subset=['SMD'])
                 .sort_values('absSMD')['label'])
df_dumb['label'] = pd.Categorical(df_dumb['label'], categories=order, ordered=True)
df_dumb = df_dumb.sort_values('label').dropna(subset=['label'])

# Plot
plt.figure(figsize=(7, max(4, 0.3*df_dumb.shape[0])))
y = np.arange(df_dumb.shape[0])

for i, r in enumerate(df_dumb.itertuples()):
    x0, x1 = r.SMD_pre, r.SMD_post
    if pd.notna(x0) and pd.notna(x1):
        plt.plot([x0, x1], [i, i], color="#bdbdbd", linewidth=5, alpha = 0.3, zorder=1)   # connector
    if pd.notna(x0):
        plt.scatter(x0, i, c="#888888", s=28, label="Pre-match" if i==0 else None, zorder=2)
    if pd.notna(x1):
        plt.scatter(x1, i, c="#F5A617", s=28, label="Post-match" if i==0 else None, zorder=3)

plt.axvline(0, color="k", linestyle=":")
plt.axvline(0.10, color="#bbbbbb", linestyle="--"); plt.axvline(-0.10, color="#bbbbbb", linestyle="--")
plt.yticks(y, df_dumb['label'])
plt.xlabel("Standardized Mean Difference  (Female − Male)")
plt.legend(loc="lower left", frameon=True)
plt.tight_layout()
plt.show()

#### **Analyze sex differences in outcome**

In [ ]:
df_imputed.columns

In [ ]:
thr = 0.025
pairs_tight = pairs.query("gower_distance < @thr").copy()

# Keep only pairs that exist in df_imputed
pairs_use = pairs_tight[
    pairs_tight.female_id.isin(df_imputed.index) &
    pairs_tight.male_id.isin(df_imputed.index)
].copy()

# define overall general colors
col_f = "#c53f60"  # female
col_m = "#3f478f"  # male

#####**Continuous  variables**

In [ ]:
var_continuous = "Symptombeginn" # Alter, HFNA, RRsysNA, BMI, Symptombeginn,

In [ ]:
sub  = df_imputed.loc[:, [var_continuous, "Geschlecht"]].copy()

# Wide, paired frame
continuous_pairs = pd.DataFrame({
    "pair_id":   pairs_use["pair_id"].to_numpy(),
    "female_id": pairs_use["female_id"].to_numpy(),
    "male_id":   pairs_use["male_id"].to_numpy(),
    "female_value": sub.loc[pairs_use["female_id"], var_continuous].to_numpy(),
    "male_value":   sub.loc[pairs_use["male_id"],   var_continuous].to_numpy(),
})

# Remove 2 extreme outliers
#continuous_pairs = continuous_pairs[(continuous_pairs["female_value"] < 20) & (continuous_pairs["male_value"] < 20)].copy()
#continuous_pairs = continuous_pairs[(continuous_pairs["female_value"] > 1000) & (continuous_pairs["male_value"] >1000)].copy()

x_f, x_m = np.zeros(len(continuous_pairs)), np.ones(len(continuous_pairs))

# Colors + jitter
jitter = 0.03
pos_f, pos_m = -0.15, 1.15
rng = np.random.default_rng(42)

# Clean vectors
f_vals = pd.to_numeric(continuous_pairs["female_value"], errors="coerce").to_numpy()
m_vals = pd.to_numeric(continuous_pairs["male_value"],   errors="coerce").to_numpy()
valid = np.isfinite(f_vals) & np.isfinite(m_vals)
f_clean = f_vals[np.isfinite(f_vals)]
m_clean = m_vals[np.isfinite(m_vals)]

fig, ax = plt.subplots(figsize=(2.5,4))

# Paired connectors with jitter
x_f = rng.normal(0.0, jitter, valid.sum())
x_m = rng.normal(1.0, jitter, valid.sum())
for xf, xm, yf, ym in zip(x_f, x_m, f_vals[valid], m_vals[valid]):
    ax.plot([xf, xm], [yf, ym], color="0.75", linewidth=0.6, alpha=0.3, zorder=1)

# Jittered points
ax.scatter(rng.normal(0.0, jitter, f_clean.size), f_clean,
           c=col_f, s=20, alpha=0.8, label="Female", zorder=2)
ax.scatter(rng.normal(1.0, jitter, m_clean.size), m_clean,
           c=col_m, s=20, alpha=0.8, label="Male",   zorder=2)

# --- Boxplots (draw last) and push zorder high so they're on top ---
bp_f = ax.boxplot([f_clean], positions=[0], widths=0.12, vert=True, boxprops=dict(color="black", facecolor="white"), medianprops=dict(color="black"),capprops=dict(visible=False),
                  patch_artist=True, showfliers=False)
bp_m = ax.boxplot([m_clean], positions=[1], widths=0.12, vert=True, boxprops=dict(color="black", facecolor="white"), medianprops=dict(color="black"),capprops=dict(visible=False),
                  patch_artist=True, showfliers=False)

ax.tick_params(axis='x', which='both', bottom=False, top=False, labelbottom=False)
ax.set_xlim(pos_f - 0.4, pos_m + 0.4)
ax.set_ylabel(f"{var_continuous}")
plt.tight_layout()
plt.show()

In [ ]:
from scipy import stats

f = pd.to_numeric(continuous_pairs["female_value"], errors="coerce")
m = pd.to_numeric(continuous_pairs["male_value"],   errors="coerce")

mask = f.notna() & m.notna()
diff = (f[mask] - m[mask]).to_numpy()
n = diff.size

# Paired t-test
tres = stats.ttest_rel(f[mask], m[mask])
# Wilcoxon (nonparametric)
wres = stats.wilcoxon(f[mask], m[mask], zero_method="wilcox", alternative="two-sided")

# Mean diff CI
se = diff.std(ddof=1) / np.sqrt(n)
tcrit = stats.t.ppf(0.975, n-1)
mean_diff = diff.mean()
ci = (mean_diff - tcrit*se, mean_diff + tcrit*se)

# Paired Cohen's dz
cohen_dz = mean_diff / diff.std(ddof=1)

print(f"Pairs: {n}")
print(f"Mean difference (Female − Male): {mean_diff:.2g}  (95% CI {ci[0]:.2g} to {ci[1]:.2g})")
#print(f"Paired t-test p = {tres.pvalue:.3g}")
print(f"Wilcoxon signed-rank p = {wres.pvalue:.3g}")
print(f"Cohen's dz = {cohen_dz:.3g}")

#####**Categorical variables**

In [ ]:
var_categorical = "EKGLSB_any"  # EKGSTHebung, EKGSTSenkungen, EKGTneg, EKGLSB_any,

In [ ]:
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.proportion import proportion_confint

sub = df_imputed.loc[:, [var_categorical, "Geschlecht"]].copy()
category_pairs = pd.DataFrame({
    "pair_id":     pairs_use["pair_id"].to_numpy(),
    "female_id":   pairs_use["female_id"].to_numpy(),
    "male_id":     pairs_use["male_id"].to_numpy(),
    "female_out":  sub.loc[pairs_use["female_id"], var_categorical].to_numpy(),
    "male_out":    sub.loc[pairs_use["male_id"],   var_categorical].to_numpy(),
})

# --- Clean to 0/1; drop pairs with missing outcome on either side ---
fo_raw = pd.to_numeric(pd.Series(category_pairs["female_out"]), errors="coerce")
mo_raw = pd.to_numeric(pd.Series(category_pairs["male_out"]),   errors="coerce")
mask = fo_raw.notna() & mo_raw.notna()
f_o = (fo_raw[mask] > 0).astype(int)   # positive -> 1, else -> 0
m_o = (mo_raw[mask] > 0).astype(int)

rate_f, rate_m = f_o.mean()*100, m_o.mean()*100

# --- Plot ---
plt.figure(figsize=(2.5,4))
plt.bar(["Female","Male"], [rate_f, rate_m], color=[col_f, col_m])
plt.ylabel(f"{var_categorical} (%)")
plt.xticks([])
plt.tight_layout()
plt.show()

In [ ]:
# 2x2 table and McNemar test
tab = pd.crosstab(f_o, m_o).reindex(index=[0,1], columns=[0,1], fill_value=0)

# Cells
a = int(tab.loc[0,0])
b = int(tab.loc[0,1])  # Male yes / Female no
c = int(tab.loc[1,0])  # Female yes / Male no
d = int(tab.loc[1,1])

# Totals
D = b + c                               # discordant pairs
N = int(tab.to_numpy().sum())           # total non-missing paired observations
# (equivalently: N = a + b + c + d)

mc = mcnemar(tab, exact=(D <= 25))
p_mc = mc.pvalue

# Matched OR (Female/Male) via discordant pairs
if D > 0:
    p_hat = c / D
    ci_low_p, ci_upp_p = proportion_confint(c, D, alpha=0.05, method="beta")
    or_hat = (p_hat / (1 - p_hat)) if 0 < p_hat < 1 else np.inf
    or_ci  = (ci_low_p/(1-ci_low_p), ci_upp_p/(1-ci_upp_p))
else:
    or_hat, or_ci = np.nan, (np.nan, np.nan)

# Absolute risk difference (Female - Male)
rd = (c - b) / N

# 95% CI via transforming the exact CI on p = c / (b+c):
if D > 0:
    rd_ci_low = (2*ci_low_p - 1) * (D / N)
    rd_ci_upp = (2*ci_upp_p - 1) * (D / N)
else:
    rd_ci_low = rd_ci_upp = np.nan

print("2x2 (Female vs Male):\n", tab)
print(f"Pairs analyzed (complete): {N}  |  Concordant a={a}, d={d}  |  Discordant b={b}, c={c} (D={D})")
print(f"McNemar p = {p_mc:.4g}")
print(f"Matched OR (Female/Male) ≈ {or_hat:.3g}  95% CI [{or_ci[0]:.3g}, {or_ci[1]:.3g}]")
print(f"Absolute risk difference (Female − Male): {rd*100:.1f} pp "
      f"(95% CI {rd_ci_low*100:.1f} to {rd_ci_upp*100:.1f} pp)")

#####**Ordinal variables**

In [ ]:
var_ordinal = "SchmerzNA"

In [ ]:
df_imputed['SchmerzNA'].value_counts()

In [ ]:
from matplotlib.patches import Patch

# Pull from full DB for just the matched IDs you kept (pairs_use)
sub = df_imputed.loc[:, [var_ordinal, "Geschlecht"]].copy()

ord_pairs = pd.DataFrame({
    "pair_id":    pairs_use["pair_id"].to_numpy(),
    "female_id":  pairs_use["female_id"].to_numpy(),
    "male_id":    pairs_use["male_id"].to_numpy(),
    "female_val": sub.loc[pairs_use["female_id"], var_ordinal].to_numpy(),
    "male_val":   sub.loc[pairs_use["male_id"],   var_ordinal].to_numpy(),
})

# Clean to integers 1..4 and drop pairs with missing
f_raw = pd.to_numeric(pd.Series(ord_pairs["female_val"]), errors="coerce")
m_raw = pd.to_numeric(pd.Series(ord_pairs["male_val"]),   errors="coerce")
mask = f_raw.notna() & m_raw.notna()
f = f_raw[mask].astype(int).clip(0, 10)
m = m_raw[mask].astype(int).clip(0, 10)

cats = [0, 1, 2, 3, 4,5,6,7,8,9,10]
n = len(f)

# proportions per category
f_counts = f.value_counts().reindex(cats, fill_value=0)
m_counts = m.value_counts().reindex(cats, fill_value=0)
f_pct = (f_counts / n * 100).to_numpy()
m_pct = (m_counts / n * 100).to_numpy()

# transparency per category (light→dark for 4→1)
alphas = [1, 0.9, 0.8, 0.7, 0.6, 0.5, 0.4, 0.3, 0.2, 0.1, 0.05] # LVEF=4 highest transparency=lightest, LVEF=1 is darkest

fig, ax = plt.subplots(figsize=(2.5, 4))

# stack bars
bottom_f = 0.0
bottom_m = 0.0
for i, (pF, pM) in enumerate(zip(f_pct, m_pct), start=1):
    ax.bar(0, pF, bottom=bottom_f, color=col_f, alpha=alphas[i-1], width=0.6, edgecolor='white', linewidth=0.6)
    ax.bar(1, pM, bottom=bottom_m, color=col_m, alpha=alphas[i-1], width=0.6, edgecolor='white', linewidth=0.6)
    bottom_f += pF
    bottom_m += pM
sex_handles = [Patch(facecolor=col_f, label="Female"),
               Patch(facecolor=col_m, label="Male")]


# plot
ax.set_xticks([0, 1]); ax.set_xticklabels(["Female", "Male"])
ax.set_ylabel(f"{var_ordinal} distribution (%)")
ax.set_ylim(0, 100)
plt.xticks([])
plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import wilcoxon

# Treat 1<2<3<4 as ordered scale; Wilcoxon signed-rank on coded levels
w = wilcoxon(f.to_numpy(), m.to_numpy(), zero_method="wilcox", alternative="two-sided")
print(f"Wilcoxon signed-rank (ordinal, Female − Male): p = {w.pvalue:.4g}  (n={n} pairs)")

# Shift table: Female−Male category differences
diff = f.to_numpy() - m.to_numpy()
shift = pd.Series(diff).value_counts().sort_index()
print("\nShift table (Female − Male category):")
print(shift.to_string())

# Also show marginal distributions (counts)
print("\nCounts per category (Female):")
print(f_counts.to_string())
print("\nCounts per category (Male):")
print(m_counts.to_string())

# show mean category difference
diff = (f - m).to_numpy()
delta = diff.mean()
rng = np.random.default_rng(42)
boot = np.array([rng.choice(diff, size=diff.size, replace=True).mean() for _ in range(10000)]) # 10,000x bootstrapping
ci_lo, ci_hi = np.quantile(boot, [0.025, 0.975])
print(f"\nMean category shift (Female−Male): {delta:.3f}  (95% CI {ci_lo:.3f} to {ci_hi:.3f})")

#### **Rev1: Create table with results (to check different caliper thresholds)**

In [ ]:
# define all thresholds and variables of interest
thresholds = [0.010, 0.025, 0.040]

continuous_vars = [
    "Alter", "HFNA", "RRsysNA", "BMI", "Symptombeginn"
]

categorical_vars = [
    "EKGSTHebung", "EKGSTSenkungen", "EKGTneg", "EKGLSB_any"
]

ordinal_vars = [
    "SchmerzNA"
]

# keep only columns that actually exist
continuous_vars = [v for v in continuous_vars if v in df_imputed.columns]
categorical_vars = [v for v in categorical_vars if v in df_imputed.columns]
ordinal_vars    = [v for v in ordinal_vars    if v in df_imputed.columns]


# helper: get matched pairs for a single threshold
def get_pairs_use(pairs, df_source, thr):
    pairs_tight = pairs.query("gower_distance < @thr").copy()
    pairs_use = pairs_tight[
        pairs_tight.female_id.isin(df_source.index) &
        pairs_tight.male_id.isin(df_source.index)
    ].copy()
    return pairs_use

##### Continuous variable sensitivity table

In [ ]:
def summarize_continuous_var(pairs, df_source, var, thr):
    pairs_use = get_pairs_use(pairs, df_source, thr)

    sub = df_source.loc[:, [var]].copy()
    dat = pd.DataFrame({
        "female": sub.loc[pairs_use["female_id"], var].to_numpy(),
        "male":   sub.loc[pairs_use["male_id"],   var].to_numpy(),
    })

    f = pd.to_numeric(dat["female"], errors="coerce")
    m = pd.to_numeric(dat["male"],   errors="coerce")
    mask = f.notna() & m.notna()
    f, m = f[mask], m[mask]

    if len(f) == 0:
        return {
            "threshold": thr, "variable": var, "type": "continuous",
            "pairs_total": len(pairs_use), "pairs_analyzed": 0,
            "p_value": np.nan, "delta": np.nan, "OR": np.nan
        }

    diff = (f - m).to_numpy()
    wres = stats.wilcoxon(f, m, zero_method="wilcox", alternative="two-sided")

    return {
        "threshold": thr, "variable": var, "type": "continuous",
        "pairs_total": len(pairs_use), "pairs_analyzed": len(f),
        "p_value": wres.pvalue,
        "delta": diff.mean(),
        "OR": np.nan
    }

##### Categorical variable sensitivity table

In [ ]:
from statsmodels.stats.contingency_tables import mcnemar

def summarize_categorical_var(pairs, df_source, var, thr):
    pairs_use = get_pairs_use(pairs, df_source, thr)

    sub = df_source.loc[:, [var]].copy()
    dat = pd.DataFrame({
        "female": sub.loc[pairs_use["female_id"], var].to_numpy(),
        "male":   sub.loc[pairs_use["male_id"],   var].to_numpy(),
    })

    f_raw = pd.to_numeric(dat["female"], errors="coerce")
    m_raw = pd.to_numeric(dat["male"],   errors="coerce")
    mask = f_raw.notna() & m_raw.notna()

    f = (f_raw[mask] > 0).astype(int)
    m = (m_raw[mask] > 0).astype(int)

    if len(f) == 0:
        return {
            "threshold": thr, "variable": var, "type": "categorical",
            "pairs_total": len(pairs_use), "pairs_analyzed": 0,
            "p_value": np.nan, "delta": np.nan, "OR": np.nan
        }

    tab = pd.crosstab(f, m).reindex(index=[0,1], columns=[0,1], fill_value=0)
    b = int(tab.loc[0,1])  # male only
    c = int(tab.loc[1,0])  # female only
    D = b + c

    mc = mcnemar(tab, exact=(D <= 25))

    if D > 0:
        p_hat = c / D
        OR = (p_hat / (1 - p_hat)) if 0 < p_hat < 1 else np.inf
    else:
        OR = np.nan

    return {
        "threshold": thr, "variable": var, "type": "categorical",
        "pairs_total": len(pairs_use), "pairs_analyzed": int(tab.to_numpy().sum()),
        "p_value": mc.pvalue,
        "delta": np.nan,
        "OR": OR
    }

##### Ordinal variable sensitivity table

In [ ]:
from scipy.stats import wilcoxon

def summarize_ordinal_var(pairs, df_source, var, thr):
    pairs_use = get_pairs_use(pairs, df_source, thr)

    sub = df_source.loc[:, [var]].copy()
    dat = pd.DataFrame({
        "female": sub.loc[pairs_use["female_id"], var].to_numpy(),
        "male":   sub.loc[pairs_use["male_id"],   var].to_numpy(),
    })

    f_raw = pd.to_numeric(dat["female"], errors="coerce")
    m_raw = pd.to_numeric(dat["male"],   errors="coerce")
    mask = f_raw.notna() & m_raw.notna()

    f = f_raw[mask].astype(int)
    m = m_raw[mask].astype(int)

    if len(f) == 0:
        return {
            "threshold": thr, "variable": var, "type": "ordinal",
            "pairs_total": len(pairs_use), "pairs_analyzed": 0,
            "p_value": np.nan, "delta": np.nan, "OR": np.nan
        }

    w = wilcoxon(f.to_numpy(), m.to_numpy(), zero_method="wilcox", alternative="two-sided")
    delta = (f - m).mean()

    return {
        "threshold": thr, "variable": var, "type": "ordinal",
        "pairs_total": len(pairs_use), "pairs_analyzed": len(f),
        "p_value": w.pvalue,
        "delta": delta,
        "OR": np.nan
    }

##### Merge variables into one table

In [ ]:
rows = []

for thr in thresholds:
    for var in continuous_vars:
        rows.append(summarize_continuous_var(pairs, df_imputed, var, thr))
    for var in categorical_vars:
        rows.append(summarize_categorical_var(pairs, df_imputed, var, thr))
    for var in ordinal_vars:
        rows.append(summarize_ordinal_var(pairs, df_imputed, var, thr))

sens_all_long = pd.DataFrame(rows)
sens_all_long

In [ ]:
# one wide table for better comparability
sens_all_wide = sens_all_long.pivot(
    index=["type", "variable"],
    columns="threshold",
    values=["pairs_total", "pairs_analyzed", "p_value", "delta", "OR"]
)

sens_all_wide.columns = [f"{metric}_thr{thr}" for metric, thr in sens_all_wide.columns]
sens_all_wide = sens_all_wide.reset_index()

sens_all_wide

In [ ]:
# round
sens_all_table = sens_all_wide.copy()

for c in sens_all_table.columns:
    if c.startswith("p_value_"):
        sens_all_table[c] = sens_all_table[c].map(lambda x: f"{x:.3g}" if pd.notna(x) else "")
    elif c.startswith("delta_"):
        sens_all_table[c] = sens_all_table[c].map(lambda x: f"{x:.2f}" if pd.notna(x) else "")
    elif c.startswith("OR_"):
        sens_all_table[c] = sens_all_table[c].map(
            lambda x: ("∞" if np.isinf(x) else f"{x:.2f}") if pd.notna(x) else ""
        )
    elif c.startswith("pairs_"):
        sens_all_table[c] = sens_all_table[c].map(lambda x: int(x) if pd.notna(x) else "")

sens_all_table

##### Display SMD results in one table

In [ ]:
def summarize_smd_threshold(pairs, dfm, sex, vars_match, cat_cols, thr):
    pairs_use = get_pairs_use(pairs, dfm, thr)

    ids_f = pd.Index(pairs_use["female_id"])
    ids_m = pd.Index(pairs_use["male_id"])
    keep_ids = ids_f.union(ids_m).astype(dfm.index.dtype).intersection(dfm.index)

    post_X   = dfm.loc[keep_ids, vars_match]
    post_sex = sex.loc[keep_ids]

    post_bal = balance_table(post_X, post_sex, vars_match, cat_cols).copy()
    post_bal = drop_zero_levels(post_bal)
    post_bal = post_bal.dropna(subset=["SMD"])

    return {
        "threshold": thr,
        "pairs_total": len(pairs_use),
        "mean_abs_SMD": post_bal["absSMD"].mean(),
        "median_abs_SMD": post_bal["absSMD"].median(),
        "max_abs_SMD": post_bal["absSMD"].max(),
        "n_covariates_gt_0.1": int((post_bal["absSMD"] > 0.1).sum())
    }

smd_rows = [
    summarize_smd_threshold(pairs, dfm, sex, vars_match, cat_cols, thr)
    for thr in thresholds
]

smd_table = pd.DataFrame(smd_rows)
smd_table

In [ ]:
smd_table_fmt = smd_table.copy()
for c in ["mean_abs_SMD", "median_abs_SMD", "max_abs_SMD"]:
    smd_table_fmt[c] = smd_table_fmt[c].map(lambda x: f"{x:.3f}" if pd.notna(x) else "")
smd_table_fmt